In [1]:
# ============ Shared configuration ============
import numpy as np
import time

p, rr, risk = 0.4054, 1.92, 0.02
initial, T  = 10_000, 8
threshold   = 0.05
z           = 2.5758293035489004

def calculate_max_dd(wins):
    rets   = np.where(wins, risk * rr, -risk)
    equity = initial * np.cumprod(1 + rets, axis=1)
    peaks  = np.maximum.accumulate(
        np.column_stack([np.full(len(wins), initial), equity]), axis=1
    )[:, 1:]
    return (1 - equity / peaks).max(axis=1)

def has_3_losses(wins):
    """True if path contains 3 consecutive losses."""
    is_loss = ~wins
    out = np.zeros(wins.shape[0], dtype=bool)
    for i in range(T - 2):
        out |= (is_loss[:, i] & is_loss[:, i+1] & is_loss[:, i+2])
    return out

# ============ Classical Monte Carlo ============
N_mc = 200_000
rng  = np.random.default_rng(42)
wins_mc = rng.random((N_mc, T)) < p

hit_maxdd_mc = calculate_max_dd(wins_mc) > threshold
hit_3loss_mc = has_3_losses(wins_mc)

p_maxdd_mc = hit_maxdd_mc.mean()
p_3loss_mc = hit_3loss_mc.mean()

print("=" * 60)
print(f"Classical Monte Carlo (T={T}, threshold={threshold:.0%})")
print("=" * 60)
print(f"Paths                    : {N_mc:,}")
print(f"P(max DD > {threshold:.0%})         = {p_maxdd_mc:.4%}")
print(f"P(3 consecutive losses)  = {p_3loss_mc:.4%}")

# ============ Exact enumeration (2^8 = 256) ============
states      = ((np.arange(2**T)[:, None] >> np.arange(T)) & 1).astype(bool)
state_probs = np.prod(np.where(states, p, 1 - p), axis=1)

exact_dd    = calculate_max_dd(states)
exact_3loss = has_3_losses(states)

p_maxdd_exact = (state_probs * (exact_dd > threshold)).sum()
p_3loss_exact = (state_probs * exact_3loss).sum()

print("\n" + "=" * 60)
print("Exact enumeration")
print("=" * 60)
print(f"Total paths              : {2**T}")
print(f"Paths with maxDD>{threshold:.0%}   : {int((exact_dd > threshold).sum())}")
print(f"Paths with 3+ losses     : {int(exact_3loss.sum())}")
print(f"P(max DD > {threshold:.0%})         = {p_maxdd_exact:.6%}")
print(f"P(3 consecutive losses)  = {p_3loss_exact:.6%}")
print(f"Equivalence check (diff) : {abs(p_maxdd_exact - p_3loss_exact):.2e}")

# ============ Real quantum oracle: 3 consecutive losses ============
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def build_oracle_3loss(T):
    """
    Phase oracle: -1 on basis states with 3+ consecutive 0s (losses).
    Real quantum circuit: computes the condition, applies phase, uncomputes.
    Ancillas returned to |0>.
    Qubits: 0..T-1 = W; T..T+n_trip-1 = triplet ancillas; T+n_trip = flag.
    """
    n_trip  = T - 2
    n_total = T + n_trip + 1
    qc = QuantumCircuit(n_total)

    W    = list(range(T))
    q    = list(range(T, T + n_trip))
    flag = T + n_trip

    # Step 1: q[i] = (W[i]=0) AND (W[i+1]=0) AND (W[i+2]=0)
    for i in range(n_trip):
        qc.x(W[i]); qc.x(W[i+1]); qc.x(W[i+2])
        qc.mcx([W[i], W[i+1], W[i+2]], q[i])
        qc.x(W[i]); qc.x(W[i+1]); qc.x(W[i+2])

    # Step 2: flag = AND(~q[i]) = "no 3-loss run anywhere"
    for i in range(n_trip):
        qc.x(q[i])
    qc.mcx(q, flag)
    for i in range(n_trip):
        qc.x(q[i])

    # Step 3: phase -1 iff flag=0 (i.e., has 3-loss run)
    qc.x(flag); qc.z(flag); qc.x(flag)

    # Step 4: uncompute flag
    for i in range(n_trip):
        qc.x(q[i])
    qc.mcx(q, flag)
    for i in range(n_trip):
        qc.x(q[i])

    # Step 5: uncompute q[i]
    for i in range(n_trip):
        qc.x(W[i]); qc.x(W[i+1]); qc.x(W[i+2])
        qc.mcx([W[i], W[i+1], W[i+2]], q[i])
        qc.x(W[i]); qc.x(W[i+1]); qc.x(W[i+2])

    return qc

# ============ Build Grover circuits ============
n_trip = T - 2
n_anc  = n_trip + 1
N_q    = T + n_anc

SHOTS    = 4000
k_values = np.array([0, 1, 2, 4])
rotation = 2 * np.arcsin(np.sqrt(p))

oracle = build_oracle_3loss(T)

circuits = []
for k in k_values:
    qc = QuantumCircuit(N_q)
    for i in range(T):
        qc.ry(rotation, i)
    for _ in range(int(k)):
        qc.compose(oracle, range(N_q), inplace=True)
        for i in range(T):
            qc.ry(-rotation, i)
        for i in range(T):
            qc.x(i)
        qc.mcp(np.pi, list(range(T - 1)), T - 1)
        for i in range(T):
            qc.x(i)
        for i in range(T):
            qc.ry(rotation, i)
    qc.measure_all()
    circuits.append(qc)

print("\n" + "=" * 60)
print("Quantum QAE (real oracle, no classical table)")
print("=" * 60)
print(f"Total qubits : {N_q}  ({T} path + {n_trip} triplet + 1 flag)")

simulator = AerSimulator(method="statevector")

print("Transpiling...")
t0 = time.time()
circuits_t = transpile(circuits, backend=simulator,
                       optimization_level=2, seed_transpiler=2026)
print(f"  done in {time.time()-t0:.1f}s")

print("Running...")
t0 = time.time()
result = simulator.run(circuits_t, shots=SHOTS, seed_simulator=2026).result()
print(f"  done in {time.time()-t0:.1f}s")

def count_3loss(counts_dict):
    total = 0
    for bs, cnt in counts_dict.items():
        w_bits = bs[::-1][:T]   # W[0]..W[T-1] in order
        w = [c == '1' for c in w_bits]
        if any((not w[i]) and (not w[i+1]) and (not w[i+2])
               for i in range(T - 2)):
            total += cnt
    return total

bad_counts = np.array([count_3loss(result.get_counts(i))
                       for i in range(len(k_values))])

print(f"k values         : {k_values}")
print(f"Bad-state counts : {bad_counts}")
print(f"Shots            : {SHOTS}")

# ============ MLE ============
from scipy.optimize import minimize_scalar

def log_likelihood(theta):
    probs = np.sin((2 * k_values + 1) * theta) ** 2
    probs = np.clip(probs, 1e-12, 1 - 1e-12)
    return np.sum(bad_counts * np.log(probs)
                  + (SHOTS - bad_counts) * np.log1p(-probs))

theta_grid = np.linspace(1e-7, np.pi / 2 - 1e-7, 200_001)
ll         = np.array([log_likelihood(th) for th in theta_grid])
best       = int(np.argmax(ll))
left       = theta_grid[max(0, best - 1)]
right      = theta_grid[min(len(theta_grid) - 1, best + 1)]

if right > left:
    fit = minimize_scalar(lambda th: -log_likelihood(th),
                          bounds=(left, right), method="bounded",
                          options={"xatol": 1e-12})
    theta_hat = fit.x
else:
    theta_hat = theta_grid[best]

quantum_prob = float(np.sin(theta_hat) ** 2)

# ============ Final comparison ============
print("\n" + "=" * 60)
print("FINAL COMPARISON")
print("=" * 60)
print(f"Exact     P(max DD > {threshold:.0%})     : {p_maxdd_exact:.6%}")
print(f"Exact     P(3 consecutive losses)  : {p_3loss_exact:.6%}")
print(f"Class.MC  P(max DD > {threshold:.0%})     : {p_maxdd_mc:.6%}")
print(f"Class.MC  P(3 consecutive losses)  : {p_3loss_mc:.6%}")
print(f"Quantum   QAE estimate             : {quantum_prob:.6%}")
print()
print(f"|Quantum - Exact(maxDD)|  : {abs(quantum_prob - p_maxdd_exact):.6%}")
print(f"|Quantum - Exact(3loss)|  : {abs(quantum_prob - p_3loss_exact):.6%}")

Classical Monte Carlo (T=8, threshold=5%)
Paths                    : 200,000
P(max DD > 5%)         = 59.3095%
P(3 consecutive losses)  = 59.3095%

Exact enumeration
Total paths              : 256
Paths with maxDD>5%   : 107
Paths with 3+ losses     : 107
P(max DD > 5%)         = 59.324258%
P(3 consecutive losses)  = 59.324258%
Equivalence check (diff) : 0.00e+00

Quantum QAE (real oracle, no classical table)
Total qubits : 15  (8 path + 6 triplet + 1 flag)
Transpiling...
  done in 0.3s
Running...
  done in 0.1s
k values         : [0 1 2 4]
Bad-state counts : [2400  921 3614 3991]
Shots            : 4000

FINAL COMPARISON
Exact     P(max DD > 5%)     : 59.324258%
Exact     P(3 consecutive losses)  : 59.324258%
Class.MC  P(max DD > 5%)     : 59.309500%
Class.MC  P(3 consecutive losses)  : 59.309500%
Quantum   QAE estimate             : 59.257322%

|Quantum - Exact(maxDD)|  : 0.066936%
|Quantum - Exact(3loss)|  : 0.066936%


In [2]:
print(
    "Pathwise equivalence:",
    np.array_equal(exact_dd > threshold, exact_3loss)
)

Pathwise equivalence: True
